In [12]:
import sys
import os

# Add project root to sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import pandas as pd
import warnings
from utils.remove_correlated_features import remove_correlated_features
from utils.feature_engineering.lagged_features import get_data_lagged
from utils.perform_xgboost_selection import perform_xgboost_selection
from utils.feature_engineering.ewma_features import get_data_ewma
from utils.feature_engineering.hybrid_features import get_data_hybrid
from utils.feature_engineering.variance_inclusion import get_data_hybrid_w_var


# Feature Selection 

The aim is to create a model for prodicting points for players in fpl. Since each position gets points based on different parameters, it is decided that one regression model is to be implemented for each position (GK, DEF, MID, FWD).

The goal of this analysis is to decide on what features are relevant for each position, this in order to reduce the compute needed to create models (I run everything locally). 

Initial plan of using correlation metrict to perform feature selection is to fragile. Using XGBoost instead based on this article; 
https://medium.com/@dhanyahari07/feature-selection-using-xgboost-f0622fb70c4d

Training and feature selection is performed on data from 22/23 and 23/24 season. The final evaluation is performed on data from the 24/25 season.


##### On using XGBoost as a feature selection pipeline
XGBoost calculates three types of feature importance scores:

* Gain: Average loss reduction gained when using a feature for splitting.

* Cover: The number of times a feature is used to split data across trees weighted by training data points.

* Weight: Total number of times a feature is used to split data across all trees.

#### Feature Selection Hyperparameters

In [ ]:
TARGET = 'total_points'  
CORR_FEATURE_TRESHOLD: float = 0.75  # Maximum correlation allowed between two features
PRINT_INFO: bool = True

#### Relevant position featurs

In [ ]:
FIELD_PLAYER_RELEVANT_FEATURES = [
                'name', 'position', 'team', 'xP', 'assists', 'bonus', 'bps', 'clean_sheets', 
                'creativity', 'expected_assists', 'expected_goal_involvements', 
                'expected_goals','expected_goals_conceded', 'goals_scored', 'ict_index', 'influence', 'minutes', 
                'opponent_team', 'own_goals', 'penalties_missed', 'red_cards', 
                'selected', 'team_a_score', 'team_h_score', 'threat', 'total_points', 
                'transfers_balance', 'value', 'was_home', 
                'yellow_cards', 'GW'
                ]

GOALKEEPER_RELEVANT_FEATURES = [
                'name', 'position', 'GW', 'xP', 'bonus', 'bps', 'clean_sheets',  
                'expected_goals_conceded', 'ict_index', 'influence', 'minutes', 
                'opponent_team', 'own_goals', 'red_cards', 'saves',
                'selected', 'team_a_score', 'team_h_score', 'total_points', 
                'transfers_balance', 'value', 'was_home', 
                'yellow_cards', 
                ]

### Getting data and performing selection

### FWD

#### Initial test
Using lagged features vs ewma

#### Lagged features

In [4]:
# Get data for FWD players
warnings.filterwarnings("ignore", category=UserWarning)

fwd_data = get_data_lagged('FWD', FIELD_PLAYER_RELEVANT_FEATURES)
fwd_data = remove_correlated_features(fwd_data, CORR_FEATURE_TRESHOLD, PRINT_INFO)

target_col = fwd_data['total_points']  # Extract the target column
fwd_data = fwd_data.drop(columns=['total_points'])  # Drop the target column from the features

fwd_features = perform_xgboost_selection(fwd_data, target_col)

# Store the selected features to a CSV file
pd.DataFrame(fwd_features).to_csv("../data/features/fwd_features_lagged.csv", index=False, header=False)

# clear memory
del fwd_data, fwd_features, target_col

Found 98 pairs of highly correlated features.
Removed 45 features due to high correlation.
Original number of features; 130. Final number of features: 85
Starting XGBoost Feature Selection
Training initial model to get feature importances
Testing 83 different feature thresholds...
Thresh=0.0000, n=83, MSE=11.2796, Best Score=11.2796
Thresh=0.0000, n=83, MSE=11.2796, Best Score=11.2796
Thresh=0.0000, n=83, MSE=11.2796, Best Score=11.2796
Thresh=0.0000, n=83, MSE=11.2796, Best Score=11.2796
Thresh=0.0000, n=83, MSE=11.2796, Best Score=11.2796
Thresh=0.0000, n=83, MSE=11.2796, Best Score=11.2796
Thresh=0.0000, n=83, MSE=11.2796, Best Score=11.2796
Thresh=0.0000, n=83, MSE=11.2796, Best Score=11.2796
Thresh=0.0003, n=75, MSE=11.3469, Best Score=11.2796
Thresh=0.0004, n=74, MSE=11.3469, Best Score=11.2796
Thresh=0.0019, n=73, MSE=11.3109, Best Score=11.2796
Thresh=0.0042, n=72, MSE=11.3109, Best Score=11.2796
Thresh=0.0048, n=71, MSE=11.2850, Best Score=11.2796
Thresh=0.0052, n=70, MSE=11.2

#### ewma

In [5]:
fwd_data = get_data_ewma('FWD', FIELD_PLAYER_RELEVANT_FEATURES)
fwd_data = remove_correlated_features(fwd_data, CORR_FEATURE_TRESHOLD, PRINT_INFO)

target_col = fwd_data['total_points']  # Extract the target column
fwd_data = fwd_data.drop(columns=['total_points'])  # Drop the target column from the features

fwd_features = perform_xgboost_selection(fwd_data, target_col)

# Store the selected features to a CSV file
pd.DataFrame(fwd_features).to_csv("../data/features/fwd_features_ewma.csv", index=False, header=False)

# clear memory
del fwd_data, fwd_features, target_col

Num elements in FWD data before filter: 6967
Num elements in FWD data after filter: 2922
Found 18 pairs of highly correlated features.
Removed 7 features due to high correlation.
Original number of features; 21. Final number of features: 14
Starting XGBoost Feature Selection
Training initial model to get feature importances
Testing 12 different feature thresholds...
Thresh=0.0628, n=12, MSE=11.9595, Best Score=11.9595
Thresh=0.0672, n=11, MSE=11.8403, Best Score=11.8403
Thresh=0.0680, n=10, MSE=11.7558, Best Score=11.7558
Thresh=0.0713, n=9, MSE=11.9039, Best Score=11.7558
Thresh=0.0778, n=8, MSE=11.7669, Best Score=11.7558
Thresh=0.0779, n=7, MSE=11.6029, Best Score=11.6029
Thresh=0.0794, n=6, MSE=11.3440, Best Score=11.3440
Thresh=0.0824, n=5, MSE=11.7732, Best Score=11.3440
Thresh=0.0973, n=4, MSE=11.6966, Best Score=11.3440
Thresh=0.0982, n=3, MSE=12.0288, Best Score=11.3440
Thresh=0.1083, n=2, MSE=14.7619, Best Score=11.3440
Thresh=0.1092, n=1, MSE=10.8116, Best Score=10.8116

Opt

#### Hybrid solution
Testing a solution using the ewma features as well as the top lagged features

In [6]:
fwd_data = get_data_hybrid('FWD', FIELD_PLAYER_RELEVANT_FEATURES)
fwd_data = remove_correlated_features(fwd_data, CORR_FEATURE_TRESHOLD, PRINT_INFO)

target_col = fwd_data['total_points']  # Extract the target column
fwd_data = fwd_data.drop(columns=['total_points'])  # Drop the target column from the features

fwd_features = perform_xgboost_selection(fwd_data, target_col)

# Store the selected features to a CSV file
pd.DataFrame(fwd_features).to_csv("../data/features/fwd_features_hybrid.csv", index=False, header=False)

# clear memory
del fwd_data, fwd_features, target_col

Creating hybrid feature set for position: FWD
Num elements in FWD data before filter: 6967
Num elements in FWD data after filter: 2922
Creating EWMA features...
--- Hybrid feature set created successfully ---
Found 70 pairs of highly correlated features.
Removed 32 features due to high correlation.
Original number of features; 65. Final number of features: 33
Starting XGBoost Feature Selection
Training initial model to get feature importances
Testing 31 different feature thresholds...
Thresh=0.0000, n=31, MSE=11.3251, Best Score=11.3251
Thresh=0.0005, n=30, MSE=11.3251, Best Score=11.3251
Thresh=0.0224, n=29, MSE=11.3251, Best Score=11.3251
Thresh=0.0232, n=28, MSE=11.4231, Best Score=11.3251
Thresh=0.0242, n=27, MSE=11.4836, Best Score=11.3251
Thresh=0.0249, n=26, MSE=11.2328, Best Score=11.2328
Thresh=0.0264, n=25, MSE=11.2132, Best Score=11.2132
Thresh=0.0265, n=24, MSE=11.4078, Best Score=11.2132
Thresh=0.0268, n=23, MSE=11.5953, Best Score=11.2132
Thresh=0.0272, n=22, MSE=11.4918,

#### Variance 

In [7]:
fwd_data = get_data_hybrid_w_var('FWD', FIELD_PLAYER_RELEVANT_FEATURES)
fwd_data = remove_correlated_features(fwd_data, CORR_FEATURE_TRESHOLD, PRINT_INFO)

target_col = fwd_data['total_points']  # Extract the target column
fwd_data = fwd_data.drop(columns=['total_points'])  # Drop the target column from the features

fwd_features = perform_xgboost_selection(fwd_data, target_col)

# Store the selected features to a CSV file
pd.DataFrame(fwd_features).to_csv("../data/features/fwd_features_hybrid.csv", index=False, header=False)

# clear memory
del fwd_data, fwd_features, target_col

Creating hybrid feature set for position: FWD
Num elements in FWD data before filter: 6967
Num elements in FWD data after filter: 2922
Creating features for predicting the variance (volatility, uncertainty)...
Creating EWMA features...
Improved Hybrid feature set created successfully
Found 58 pairs of highly correlated features.
Removed 29 features due to high correlation.
Original number of features; 65. Final number of features: 36
Starting XGBoost Feature Selection
Training initial model to get feature importances
Testing 34 different feature thresholds...
Thresh=0.0020, n=34, MSE=11.3681, Best Score=11.3681
Thresh=0.0025, n=33, MSE=11.3681, Best Score=11.3681
Thresh=0.0125, n=32, MSE=11.3681, Best Score=11.3681
Thresh=0.0164, n=31, MSE=11.3334, Best Score=11.3334
Thresh=0.0207, n=30, MSE=11.2046, Best Score=11.2046
Thresh=0.0210, n=29, MSE=11.2987, Best Score=11.2046
Thresh=0.0216, n=28, MSE=11.1966, Best Score=11.1966
Thresh=0.0218, n=27, MSE=11.3652, Best Score=11.1966
Thresh=0.0

### MID

Having less features seems to do the model good, thus I use the hybrid-fetch

In [8]:
mid_data = get_data_hybrid('MID', FIELD_PLAYER_RELEVANT_FEATURES)
mid_data = remove_correlated_features(mid_data, CORR_FEATURE_TRESHOLD, PRINT_INFO)

target_col = mid_data['total_points']  # Extract the target column
mid_data = mid_data.drop(columns=['total_points'])  # Drop the target column from the features

mid_features = perform_xgboost_selection(mid_data, target_col)

# Store the selected features to a CSV file
pd.DataFrame(mid_features).to_csv("../data/features/mid_features_hybrid.csv", index=False, header=False)

# clear memory
del mid_data, mid_features, target_col

Creating hybrid feature set for position: MID
Num elements in MID data before filter: 24265
Num elements in MID data after filter: 10642
Creating EWMA features...
--- Hybrid feature set created successfully ---
Found 53 pairs of highly correlated features.
Removed 30 features due to high correlation.
Original number of features; 65. Final number of features: 35
Starting XGBoost Feature Selection
Training initial model to get feature importances
Testing 33 different feature thresholds...
Thresh=0.0113, n=33, MSE=7.8131, Best Score=7.8131
Thresh=0.0212, n=32, MSE=7.7968, Best Score=7.7968
Thresh=0.0231, n=31, MSE=7.8239, Best Score=7.7968
Thresh=0.0234, n=30, MSE=7.8450, Best Score=7.7968
Thresh=0.0235, n=29, MSE=7.8274, Best Score=7.7968
Thresh=0.0237, n=28, MSE=7.8678, Best Score=7.7968
Thresh=0.0238, n=27, MSE=7.8478, Best Score=7.7968
Thresh=0.0242, n=26, MSE=7.8700, Best Score=7.7968
Thresh=0.0242, n=25, MSE=7.8469, Best Score=7.7968
Thresh=0.0244, n=24, MSE=7.8685, Best Score=7.796

### DEF

In [9]:
def_data = get_data_hybrid('DEF', FIELD_PLAYER_RELEVANT_FEATURES)
def_data = remove_correlated_features(def_data, CORR_FEATURE_TRESHOLD, PRINT_INFO)

target_col = def_data['total_points']  # Extract the target column
def_data = def_data.drop(columns=['total_points'])  # Drop the target column from the features

def_features = perform_xgboost_selection(def_data, target_col)

# Store the selected features to a CSV file
pd.DataFrame(def_features).to_csv("../data/features/def_features_hybrid.csv", index=False, header=False)

# clear memory
del def_data, def_features, target_col

Creating hybrid feature set for position: DEF
Num elements in DEF data before filter: 18794
Num elements in DEF data after filter: 7619
Creating EWMA features...
--- Hybrid feature set created successfully ---
Found 30 pairs of highly correlated features.
Removed 21 features due to high correlation.
Original number of features; 65. Final number of features: 44
Starting XGBoost Feature Selection
Training initial model to get feature importances
Testing 42 different feature thresholds...
Thresh=0.0000, n=42, MSE=8.5093, Best Score=8.5093
Thresh=0.0072, n=41, MSE=8.5093, Best Score=8.5093
Thresh=0.0174, n=40, MSE=8.5217, Best Score=8.5093
Thresh=0.0177, n=39, MSE=8.5296, Best Score=8.5093
Thresh=0.0184, n=38, MSE=8.4333, Best Score=8.4333
Thresh=0.0185, n=37, MSE=8.4719, Best Score=8.4333
Thresh=0.0188, n=36, MSE=8.4841, Best Score=8.4333
Thresh=0.0189, n=35, MSE=8.4157, Best Score=8.4157
Thresh=0.0193, n=34, MSE=8.4749, Best Score=8.4157
Thresh=0.0194, n=33, MSE=8.5567, Best Score=8.4157

#### GK

In [10]:
gk_data = get_data_hybrid('GK', FIELD_PLAYER_RELEVANT_FEATURES)
gk_data = remove_correlated_features(gk_data, CORR_FEATURE_TRESHOLD, PRINT_INFO)

target_col = gk_data['total_points']  # Extract the target column
gk_data = gk_data.drop(columns=['total_points'])  # Drop the target column from the features

gk_features = perform_xgboost_selection(gk_data, target_col)

# Store the selected features to a CSV file
pd.DataFrame(gk_features).to_csv("../data/features/gk_features_hybrid.csv", index=False, header=False)

# clear memory
del gk_data, gk_features, target_col

Creating hybrid feature set for position: GK
Num elements in GK data before filter: 6204
Num elements in GK data after filter: 1546
Creating EWMA features...
--- Hybrid feature set created successfully ---
Found 34 pairs of highly correlated features.
Removed 22 features due to high correlation.
Original number of features; 65. Final number of features: 43
Starting XGBoost Feature Selection
Training initial model to get feature importances
Testing 41 different feature thresholds...
Thresh=0.0000, n=41, MSE=9.2380, Best Score=9.2380
Thresh=0.0000, n=41, MSE=9.2380, Best Score=9.2380
Thresh=0.0000, n=41, MSE=9.2380, Best Score=9.2380
Thresh=0.0000, n=41, MSE=9.2380, Best Score=9.2380
Thresh=0.0000, n=41, MSE=9.2380, Best Score=9.2380
Thresh=0.0000, n=41, MSE=9.2380, Best Score=9.2380
Thresh=0.0138, n=35, MSE=9.2380, Best Score=9.2380
Thresh=0.0140, n=34, MSE=9.2731, Best Score=9.2380
Thresh=0.0154, n=33, MSE=9.2731, Best Score=9.2380
Thresh=0.0196, n=32, MSE=9.2389, Best Score=9.2380
Thr